In [0]:
%sql
-- ΕΝΟΤΗΤΑ 1: Αρχικοποίηση Περιβάλλοντος και Δημιουργία Σχημάτων (Medallion Setup)
CREATE WIDGET TEXT catalog DEFAULT "bootcamp";
CREATE WIDGET TEXT bronze_schema DEFAULT "bronze_marios_urban";
CREATE WIDGET TEXT silver_schema DEFAULT "silver_marios_urban";
CREATE WIDGET TEXT gold_schema DEFAULT "gold_marios_urban";
CREATE WIDGET TEXT volume_landing_path DEFAULT "/Volumes/bootcamp/bronze_marios_urban/landing";

In [0]:
%sql
/*
  Phase 3: Gold: 
  3.16 Δημιουργία dim_date (2024-01-01 έως 2025-12-31)
  Χρήση sequence + explode σύμφωνα με τις απαιτήσεις του project.
*/

CREATE OR REPLACE TABLE bootcamp.gold_marios_urban.dim_date AS
WITH date_series AS (
  SELECT explode(sequence(to_date('2024-01-01'), to_date('2025-12-31'), interval 1 day)) AS full_date
)
SELECT
  cast(date_format(full_date, 'yyyyMMdd') AS INT) AS date_key,
  full_date AS date,
  year(full_date) AS year,
  quarter(full_date) AS quarter,
  month(full_date) AS month,
  date_format(full_date, 'MMMM') AS month_name,
  day(full_date) AS day_of_month,
  dayofweek(full_date) AS day_of_week,
  date_format(full_date, 'EEEE') AS day_name,
  CASE WHEN dayofweek(full_date) IN (1, 7) THEN true ELSE false END AS is_weekend
FROM date_series;

-- Έλεγχος: Περιμένουμε ακριβώς 731 ημέρες (366 για το δίσεκτο 2024 + 365 για το 2025)
SELECT count(*) AS total_days FROM bootcamp.gold_marios_urban.dim_date;

In [0]:
%sql
/*
  3.17: Δημιουργία dim_customer
  - Surrogate key: customer_key (IDENTITY)
  - Unknown member: customer_key = -1
  - Primary Key constraint
*/

CREATE OR REPLACE TABLE bootcamp.gold_marios_urban.dim_customer (
  customer_key BIGINT GENERATED BY DEFAULT AS IDENTITY,
  customer_id STRING,
  first_name STRING,
  last_name STRING,
  email STRING,
  gender STRING,
  birth_date DATE,
  city STRING,
  region STRING,
  segment STRING,
  signup_date DATE,
  CONSTRAINT pk_dim_customer PRIMARY KEY (customer_key)
);

-- 1. Unknown Member (-1)
INSERT INTO bootcamp.gold_marios_urban.dim_customer (
  customer_key, customer_id, first_name, last_name, email, gender, birth_date, city, region, segment, signup_date
)
VALUES (
  -1, 'UNKNOWN', 'Unknown', 'Customer', 'unknown@domain.com', 'U', NULL, 'Unknown', 'Unknown', 'Unknown', NULL
);

-- 2. Καθαροί πελάτες από Silver
INSERT INTO bootcamp.gold_marios_urban.dim_customer (
  customer_id, first_name, last_name, email, gender, birth_date, city, region, segment, signup_date
)
SELECT 
  customer_id,
  first_name,
  last_name,
  email,
  gender,
  birth_date,
  city,
  region,
  segment,
  signup_date
FROM bootcamp.silver_marios_urban.slv_customers;

-- Έλεγχος: Περιμένουμε 486 εγγραφές
SELECT count(*) AS total_customers FROM bootcamp.gold_marios_urban.dim_customer;

In [0]:
%sql
/*
  3.18: Δημιουργία dim_product
  - Surrogate key: product_key (IDENTITY)
  - Unknown member: product_key = -1
  - Primary Key constraint
*/

CREATE OR REPLACE TABLE bootcamp.gold_marios_urban.dim_product (
  product_key BIGINT GENERATED BY DEFAULT AS IDENTITY,
  product_id STRING,
  product_name STRING,
  category STRING,
  subcategory STRING,
  brand STRING,
  cost_price DECIMAL(10,2),
  list_price DECIMAL(10,2),
  is_active BOOLEAN,
  CONSTRAINT pk_dim_product PRIMARY KEY (product_key)
);

-- 1. Unknown Member (-1)
INSERT INTO bootcamp.gold_marios_urban.dim_product (
  product_key, product_id, product_name, category, subcategory, brand, cost_price, list_price, is_active
)
VALUES (
  -1, 'UNKNOWN', 'Unknown Product', 'Unknown', 'Unknown', 'Unknown', 0.00, 0.00, false
);

-- 2. Καθαρά προϊόντα από slv_products
INSERT INTO bootcamp.gold_marios_urban.dim_product (
  product_id, product_name, category, subcategory, brand, cost_price, list_price, is_active
)
SELECT 
  product_id,
  product_name,
  category,
  subcategory,
  brand,
  unit_cost,
  list_price,
  is_active
FROM bootcamp.silver_marios_urban.slv_products;

-- Έλεγχος: Περιμένουμε 76 εγγραφές
SELECT count(*) AS total_products FROM bootcamp.gold_marios_urban.dim_product;

In [0]:
%sql
/*
  3.19: Δημιουργία dim_store
  - Surrogate key: store_key (IDENTITY)
  - Unknown member: store_key = -1
  - Primary Key constraint
*/

CREATE OR REPLACE TABLE bootcamp.gold_marios_urban.dim_store (
  store_key BIGINT GENERATED BY DEFAULT AS IDENTITY,
  store_id STRING,
  store_name STRING,
  store_type STRING,
  city STRING,
  region STRING,
  size_sqm INT,
  open_date DATE,
  CONSTRAINT pk_dim_store PRIMARY KEY (store_key)
);

-- 1. Unknown Member (-1)
INSERT INTO bootcamp.gold_marios_urban.dim_store (
  store_key, store_id, store_name, store_type, city, region, size_sqm, open_date
)
VALUES (
  -1, 'UNKNOWN', 'Unknown Store', 'Unknown', 'Unknown', 'Unknown', NULL, NULL
);

-- 2. Καθαρά καταστήματα από slv_stores
INSERT INTO bootcamp.gold_marios_urban.dim_store (
  store_id, store_name, store_type, city, region, size_sqm, open_date
)
SELECT 
  store_id,
  store_name,
  store_type,
  city,
  region,
  size_sqm,
  open_date
FROM bootcamp.silver_marios_urban.slv_stores;

-- Έλεγχος: Περιμένουμε 14 εγγραφές
SELECT count(*) AS total_stores FROM bootcamp.gold_marios_urban.dim_store;

In [0]:
%sql
/*
  3.20: Gold fact (Βήμα 1/2): Δημιουργία δομής fact_sales
*/
CREATE OR REPLACE TABLE bootcamp.gold_marios_urban.fact_sales (
  sales_key BIGINT GENERATED BY DEFAULT AS IDENTITY,
  order_date_key INT,
  customer_key BIGINT,
  product_key BIGINT,
  store_key BIGINT,
  order_id STRING,
  order_line_id STRING,
  payment_method STRING,
  order_status STRING,
  quantity INT,
  unit_price DECIMAL(10,2),
  discount_pct DECIMAL(5,2),
  gross_amount DECIMAL(12,2),
  discount_amount DECIMAL(12,2),
  net_amount DECIMAL(12,2),
  cost_amount DECIMAL(12,2),
  margin_amount DECIMAL(12,2),
  CONSTRAINT pk_fact_sales PRIMARY KEY (sales_key)
)
CLUSTER BY (order_date_key);

In [0]:
%sql
/*
  3.21: Gold fact (Βήμα 2/2): Εισαγωγή δεδομένων & Έλεγχος
*/
INSERT INTO bootcamp.gold_marios_urban.fact_sales (
  order_date_key,
  customer_key,
  product_key,
  store_key,
  order_id,
  order_line_id,
  payment_method,
  order_status,
  quantity,
  unit_price,
  discount_pct,
  gross_amount,
  discount_amount,
  net_amount,
  cost_amount,
  margin_amount
)
SELECT 
  cast(date_format(o.order_ts, 'yyyyMMdd') AS INT) AS order_date_key,
  coalesce(c.customer_key, -1) AS customer_key,
  coalesce(p.product_key, -1) AS product_key,
  coalesce(s.store_key, -1) AS store_key,
  oi.order_id,
  oi.order_line_id,
  o.payment_method,
  o.order_status,
  oi.quantity,
  oi.unit_price,
  oi.discount_pct,
  oi.gross_amount,
  oi.discount_amount,
  oi.net_amount,
  round(oi.quantity * coalesce(p.cost_price, 0.00), 2) AS cost_amount,
  round(oi.net_amount - round(oi.quantity * coalesce(p.cost_price, 0.00), 2), 2) AS margin_amount
FROM bootcamp.silver_marios_urban.slv_order_items oi
JOIN bootcamp.silver_marios_urban.slv_orders o 
  ON oi.order_id = o.order_id
LEFT JOIN bootcamp.gold_marios_urban.dim_customer c 
  ON o.customer_id = c.customer_id
LEFT JOIN bootcamp.gold_marios_urban.dim_product p 
  ON oi.product_id = p.product_id
LEFT JOIN bootcamp.gold_marios_urban.dim_store s 
  ON o.store_id = s.store_id;

-- Έλεγχος: Περιμένουμε ακριβώς 23.998 γραμμές
SELECT count(*) AS total_fact_rows FROM bootcamp.gold_marios_urban.fact_sales;

In [0]:
%sql
--Περίμενα 23.998 γραμμές και μου εμφανίστηκαν 23.997 γραμμές, άρα μία γραμμή του slv_order_items δεν βρήκε το αντίστοιχο order_id της στον πίνακα slv_orders. Ψάχνω τη γραμμή
SELECT oi.*
FROM bootcamp.silver_marios_urban.slv_order_items oi
LEFT JOIN bootcamp.silver_marios_urban.slv_orders o 
  ON oi.order_id = o.order_id
WHERE o.order_id IS NULL;

In [0]:
%sql
--εντοπίστηκε η γραμμή L9999999 με order_id = O999999.  Πρόκειται για ορφανή εγγραφή (orphan record): υπάρχει γραμμή παραγγελίας (order_line_id = L9999999), αλλά δεν υπάρχει καμία αντίστοιχη παραγγελία με order_id = O999999 στον πίνακα των παραγγελιών. Αυτή η 1 γραμμή δεν πρέπει να βρίσκεται στο slv_order_items, αλλά να αποσταλεί στον πίνακα απορρίψεων silver.etl_rejects με αιτιολογία 'ORPHAN_ORDER_ID'. Στέλνω την εγγραφή στον πίνακα απορρίψεων.
INSERT INTO bootcamp.silver_marios_urban.etl_rejects
SELECT 
  'brz_order_items' AS source_table,
  'ORPHAN_ORDER_ID' AS reject_reason,
  concat_ws(', ', order_line_id, order_id, product_id, quantity, unit_price, discount_pct) AS rejected_record,
  current_timestamp() AS rejected_at
FROM bootcamp.silver_marios_urban.slv_order_items
WHERE order_id = 'O999999';

In [0]:
%sql
--διαγράφω την εγγραφή από το slv_order_items
DELETE FROM bootcamp.silver_marios_urban.slv_order_items
WHERE order_id = 'O999999';

-- Έλεγχος: πρέπει να εμφανίσει 23.997
SELECT count(*) AS clean_order_items FROM bootcamp.silver_marios_urban.slv_order_items;

In [0]:
%sql
--3.22: Προσθήκη Foreign Key constraints

-- dim_date δημιουργήθηκε με CTAS χωρίς PK — προσθήκη NOT NULL και PK πρώτα
ALTER TABLE bootcamp.gold_marios_urban.dim_date ALTER COLUMN date_key SET NOT NULL;

ALTER TABLE bootcamp.gold_marios_urban.fact_sales DROP CONSTRAINT IF EXISTS fk_sales_date;
ALTER TABLE bootcamp.gold_marios_urban.dim_date DROP CONSTRAINT IF EXISTS pk_dim_date;
ALTER TABLE bootcamp.gold_marios_urban.dim_date 
  ADD CONSTRAINT pk_dim_date PRIMARY KEY (date_key);

-- Οι fk_sales_customer, fk_sales_product, fk_sales_store δημιουργήθηκαν ήδη
ALTER TABLE bootcamp.gold_marios_urban.fact_sales 
  ADD CONSTRAINT fk_sales_date FOREIGN KEY (order_date_key) 
  REFERENCES bootcamp.gold_marios_urban.dim_date (date_key);

In [0]:
%sql
/*
  3.23 Quality Check:
  1. Συνολικές εγγραφές (αναμένονται 23.997)
  2. Παραγγελίες με Unknown Customer (-1)
  3. Παραγγελίες με Unknown Product (-1)
  4. Παραγγελίες με Unknown Store (-1)
  5. Έλεγχος για ανεπιθύμητα NULLs στα κλειδιά (αναμένονται 0)
*/
SELECT 
  count(*) AS total_rows,
  count(CASE WHEN customer_key = -1 THEN 1 END) AS unknown_customers,
  count(CASE WHEN product_key = -1 THEN 1 END) AS unknown_products,
  count(CASE WHEN store_key = -1 THEN 1 END) AS unknown_stores,
  count(CASE WHEN order_date_key IS NULL OR customer_key IS NULL OR product_key IS NULL OR store_key IS NULL THEN 1 END) AS null_keys
FROM bootcamp.gold_marios_urban.fact_sales;

In [0]:
%sql
/*
  Stretch Goal (Bonus): Aggregate Fact Table
  Grain: Year-Month x Store x Category
*/
CREATE OR REPLACE TABLE bootcamp.gold_marios_urban.fact_sales_monthly
CLUSTER BY (year_month, store_key)
AS
SELECT 
  concat(d.year, '-', lpad(d.month, 2, '0')) AS year_month,
  d.year,
  d.month,
  f.store_key,
  p.category,
  count(DISTINCT f.order_id) AS total_monthly_orders,
  count(f.sales_key) AS total_order_lines,
  sum(f.quantity) AS total_units_sold,
  round(sum(f.gross_amount), 2) AS total_gross_revenue,
  round(sum(f.discount_amount), 2) AS total_discounts,
  round(sum(f.net_amount), 2) AS total_net_revenue,
  round(sum(f.margin_amount), 2) AS total_margin
FROM bootcamp.gold_marios_urban.fact_sales f
JOIN bootcamp.gold_marios_urban.dim_date d ON f.order_date_key = d.date_key
JOIN bootcamp.gold_marios_urban.dim_product p ON f.product_key = p.product_key
GROUP BY 
  concat(d.year, '-', lpad(d.month, 2, '0')),
  d.year,
  d.month,
  f.store_key,
  p.category;

num_affected_rows,num_inserted_rows


In [0]:
%sql
/*
  Stretch Goal (Bonus): Junk Dimension for order attributes
*/
CREATE OR REPLACE TABLE bootcamp.gold_marios_urban.dim_order_attributes (
  attribute_key BIGINT GENERATED ALWAYS AS IDENTITY,
  payment_method STRING,
  order_status STRING
);

INSERT INTO bootcamp.gold_marios_urban.dim_order_attributes (payment_method, order_status)
SELECT DISTINCT 
  coalesce(payment_method, 'Unknown') AS payment_method,
  coalesce(order_status, 'Unknown') AS order_status
FROM bootcamp.silver_marios_urban.slv_orders

UNION ALL

SELECT 'Unknown', 'Unknown';

num_affected_rows,num_inserted_rows
10,10
